# Tahap 2: Partisi Batas Kecamatan
### JEF 2026 · Topik 4: Penguatan Infrastruktur Pendukung Ekonomi Kreatif

**Input** (dari Tahap 1): `poi_creative.gpkg`, `jakarta_studyarea.gpkg`.

**Keluaran:** `data/processed/kecamatan_bersih.gpkg`, yang dipakai Tahap 3.

Batas `admin_level=6` dari OSM yang diambil di Tahap 1 menghasilkan 200 unit, padahal DKI Jakarta memiliki 44 kecamatan (42 di daratan) dan 267 kelurahan. Penandaan `admin_level=6` di OSM Indonesia tidak konsisten: sebagian kelurahan ikut bertanda level 6, sehingga poligonnya tumpang tindih. Unit yang tumpang tindih membuat matriks bobot ketetanggaan tidak sah dan menghitung ganda POI yang sama.

SEL 2 memperbaikinya dengan algoritma partisi greedy: poligon terbesar dipertahankan, dan poligon yang bersarang di dalamnya dibuang, tanpa mengubah cakupan wilayah. Hasilnya 44 unit (42 kecamatan daratan dan 2 unit Kepulauan Seribu); Tahap 3 memakai 42 kecamatan daratan.

## SEL 1 — SETUP

In [1]:
# ============ SEL 1: SETUP ============
import os, sys, glob, sqlite3, json, warnings, time
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

# --- PROJ GUARD (sama seperti Tahap 1) ---
_cands = [os.path.join(sys.prefix, "share", "proj"),
          os.path.join(sys.prefix, "Library", "share", "proj")]
try:
    import pyproj
    _cands.append(os.path.join(os.path.dirname(pyproj.__file__), "proj_dir", "share", "proj"))
except Exception:
    pass
for c in _cands:
    if os.path.exists(os.path.join(c, "proj.db")):
        os.environ["PROJ_LIB"] = os.environ["PROJ_DATA"] = c
        try:
            import pyproj; pyproj.datadir.set_data_dir(c)
        except Exception: pass
        print("PROJ:", c)
        break
os.environ["SHAPE_RESTORE_SHX"] = "YES"

import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import rasterio, h3
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union
from scipy.stats import entropy as shannon_entropy
from sklearn.cluster import DBSCAN
from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local

print(f"geopandas {gpd.__version__} | h3 {h3.__version__}")

# --- PARAMETER ---
CRS_GEO, CRS_METRIC = 4326, 32748
H3_MAIN, H3_COARSE  = 8, 7          # resolusi utama & kasar
KEC_TARGET          = 42            # kecamatan daratan DKI (44 total - 2 Kep. Seribu)
JAKARTA_LAND_KM2    = 662.0
DBSCAN_EPS_M        = 400           # radius ketetanggaan klaster (meter)
DBSCAN_MIN_SAMPLES  = 10
SIG                 = 0.05

BASE = Path.cwd()
PROC = BASE / "data" / "processed"
FIG  = BASE / "outputs" / "figures"
TAB  = BASE / "outputs" / "tables"
for p in (FIG, TAB): p.mkdir(parents=True, exist_ok=True)

CULINARY = {"kuliner_kafe", "kuliner_restoran"}
SUBSEKTOR_LABEL = {
    "seni_pertunjukan":"Seni Pertunjukan","seni_rupa_galeri":"Seni Rupa & Galeri","musik":"Musik",
    "film_sinema":"Film & Sinema","kriya":"Kriya","fesyen":"Fesyen",
    "penerbitan_literasi":"Penerbitan & Literasi","desain_layanan_kreatif":"Desain & Layanan Kreatif",
    "coworking_hub":"Coworking / Creative Hub","fotografi":"Fotografi",
    "kuliner_kafe":"Kuliner - Kafe","kuliner_restoran":"Kuliner - Restoran"}

mpl.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                     "axes.titlesize": 11, "axes.titleweight": "bold"})

RESULTS = {"run_utc": datetime.now(timezone.utc).isoformat(), "params": {
    "h3_main": H3_MAIN, "h3_coarse": H3_COARSE, "dbscan_eps_m": DBSCAN_EPS_M,
    "dbscan_min_samples": DBSCAN_MIN_SAMPLES, "alpha": SIG}}

print("SETUP SELESAI |", PROC)


PROJ: ~/miniforge3/envs/r2c-dl/share/proj
geopandas 1.1.3 | h3 4.5.0
SETUP SELESAI | <folder-proyek>/data/processed


## SEL 2 — Muat data & perbaiki unit kecamatan

In [2]:
assert "PROC" in globals(), "Jalankan SEL 1 (SETUP) dulu."

poi = gpd.read_file(PROC / "poi_creative.gpkg", layer="poi").to_crs(CRS_GEO)
kec_raw = gpd.read_file(PROC / "jakarta_studyarea.gpkg", layer="kecamatan").to_crs(CRS_GEO)
study_area = gpd.read_file(PROC / "jakarta_studyarea.gpkg", layer="study_area").geometry.iloc[0]
print(f"Dimuat: {len(poi)} POI | {len(kec_raw)} unit batas mentah")

def dedup_partition(gdf, overlap_thresh=0.30):
    # Greedy: pertahankan poligon TERBESAR; buang yang >30% luasnya sudah tertutup unit terpilih
    g = gdf.to_crs(CRS_METRIC).copy()
    g["_a"] = g.geometry.area
    g = g.sort_values("_a", ascending=False)
    kept, acc = [], None
    for idx, row in g.iterrows():
        gm = row.geometry
        if acc is not None and gm.intersection(acc).area / max(gm.area, 1e-12) > overlap_thresh:
            continue
        kept.append(idx)
        acc = gm if acc is None else unary_union([acc, gm])
    return gdf.loc[kept].copy()

area_before = kec_raw.to_crs(CRS_METRIC).geometry.union_all().area / 1e6
kec = dedup_partition(kec_raw).reset_index(drop=True)
area_after = kec.to_crs(CRS_METRIC).geometry.union_all().area / 1e6

print(f"\nPartisi: {len(kec_raw)} -> {len(kec)} unit")
print(f"Cakupan wilayah: {area_before:,.1f} -> {area_after:,.1f} km2 (harus nyaris sama)")
print(f"Rujukan resmi: {KEC_TARGET} kecamatan daratan, {JAKARTA_LAND_KM2:.0f} km2")

if abs(len(kec) - KEC_TARGET) > 6:
    print(f"PERINGATAN: jumlah unit ({len(kec)}) masih jauh dari {KEC_TARGET}.")
    print("  Periksa daftar nama di bawah; sesuaikan overlap_thresh bila perlu.")
else:
    print("Jumlah unit: WAJAR")

kec["luas_km2"] = kec.to_crs(CRS_METRIC).area / 1e6
kec = kec.sort_values("kecamatan").reset_index(drop=True)
print(f"\nDaftar {len(kec)} unit (periksa terhadap daftar resmi 42 kecamatan):")
for i, r in kec.iterrows():
    print(f"  {i+1:2d}. {str(r['kecamatan'])[:28]:30s} {r['luas_km2']:7.2f} km2   [{str(r['kabkota'])[:22]}]")

kec.to_file(PROC / "kecamatan_bersih.gpkg", driver="GPKG")
RESULTS["kecamatan"] = {"n_raw": int(len(kec_raw)), "n_clean": int(len(kec)),
                        "area_km2": round(float(area_after), 1)}


Dimuat: 2924 POI | 200 unit batas mentah

Partisi: 200 -> 44 unit
Cakupan wilayah: 648.9 -> 648.9 km2 (harus nyaris sama)
Rujukan resmi: 42 kecamatan daratan, 662 km2
Jumlah unit: WAJAR

Daftar 44 unit (periksa terhadap daftar resmi 42 kecamatan):
   1. Cakung                           40.40 km2   [Jakarta Timur]
   2. Cempaka Putih                     4.71 km2   [Jakarta Pusat]
   3. Cengkareng                       25.69 km2   [Jakarta Barat]
   4. Cilandak                         17.55 km2   [Jakarta Selatan]
   5. Cilincing                        42.07 km2   [Jakarta Utara]
   6. Cipayung                         28.54 km2   [Jakarta Timur]
   7. Ciracas                          16.54 km2   [Jakarta Timur]
   8. Duren Sawit                      22.02 km2   [Jakarta Timur]
   9. Gambir                            7.49 km2   [Jakarta Pusat]
  10. Grogol Petamburan                10.73 km2   [Jakarta Barat]
  11. Jagakarsa                        25.14 km2   [Jakarta Selatan]
  12. Jatin